# Imports

In [1]:
import numpy as np
import pandas as pd

import os
import pickle
import datetime
import time

import random

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wilcoxon

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_curve, roc_auc_score, average_precision_score, precision_recall_curve, RocCurveDisplay, accuracy_score

import prepare_data
import cutpoint_analysis

In [2]:
from set_env_vars import set_all_env_vars

set_all_env_vars()

# Load data

In [1]:
train_cohort_ll = prepare_data.load_and_process_cohort('train', 'latest')
val_cohort_ll = prepare_data.load_and_process_cohort('val', 'latest')

train_cohort_med = prepare_data.load_and_process_cohort('train', 'median')
val_cohort_med = prepare_data.load_and_process_cohort('val', 'median')

# Load results of feature ll_feature_set_diction

In [4]:
def get_feature_set_union_and_intersect(feature_set_list, return_as_dict=False):
    union_of_sets = []

    for i, fs in enumerate(feature_set_list):
        print('Set %d has length %d' % (i+1, len(fs)))
        for feature in fs:
            if feature not in union_of_sets:
                union_of_sets.append(feature)

    print('Union set has length %d' % len(union_of_sets))
    
    intersection_of_sets = [
        feature for feature in feature_set_list[0] if (
            all([feature in feature_set_list[i] for i in range(len(feature_set_list))])
        )
    ]

    print('Intersection set has length %d' % len(intersection_of_sets))
    
    if return_as_dict:
        feature_set_dict = {
            'union': union_of_sets,
            'intersection': intersection_of_sets
        }
    
    return union_of_sets, intersection_of_sets

In [5]:
ll_feature_set_dict = dict()
med_feature_set_dict = dict()

# Latest lab imputation

## Logistic regression only

### 3 iter

with open('pickle/trimmed_feature_sets/ll_auroc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_roc_logreg_only_intersect'] = intersect
    ll_feature_set_dict['ll_roc_logreg_only_union'] = union

with open('pickle/trimmed_feature_sets/ll_auprc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_prc_logreg_only_intersect'] = intersect
    ll_feature_set_dict['ll_prc_logreg_only_union'] = union
    
## SVC and logistic regression

with open('pickle/trimmed_feature_sets/ll_auroc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_roc_svc_intersect'] = intersect
    ll_feature_set_dict['ll_roc_svc_union'] = union

with open('pickle/trimmed_feature_sets/ll_auprc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_prc_svc_intersect'] = intersect
    ll_feature_set_dict['ll_prc_svc_union'] = union
    
    
# Median imputation

## Logistic regression only

### 3 iter
    
with open('pickle/trimmed_feature_sets/med_auroc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_roc_logreg_only_intersect'] = intersect
    med_feature_set_dict['med_roc_logreg_only_union'] = union

with open('pickle/trimmed_feature_sets/med_auprc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_prc_logreg_only_intersect'] = intersect
    med_feature_set_dict['med_prc_logreg_only_union'] = union
    
## SVC and logistic regression

with open('pickle/trimmed_feature_sets/med_auroc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_roc_svc_intersect'] = intersect
    med_feature_set_dict['med_roc_svc_union'] = union

with open('pickle/trimmed_feature_sets/med_auprc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_prc_svc_intersect'] = intersect
    med_feature_set_dict['med_prc_svc_union'] = union

Set 1 has length 36
Set 2 has length 36
Set 3 has length 36
Union set has length 36
Intersection set has length 36
Set 1 has length 38
Set 2 has length 38
Set 3 has length 38
Union set has length 38
Intersection set has length 38
Set 1 has length 37
Union set has length 37
Intersection set has length 37
Set 1 has length 55
Union set has length 55
Intersection set has length 55
Set 1 has length 52
Set 2 has length 56
Set 3 has length 58
Union set has length 59
Intersection set has length 51
Set 1 has length 52
Set 2 has length 56
Set 3 has length 58
Union set has length 59
Intersection set has length 51
Set 1 has length 44
Union set has length 44
Intersection set has length 44
Set 1 has length 56
Union set has length 56
Intersection set has length 56


# Define functions to use for assessing model performance

## Assess model performance on input model type/feature set/dataset

In [16]:
def assess_model_on_features(train_df,
                             test_df,
                             feature_set,
                             model_callable,
                             params_dict,
                             feature_set_label,
                             metric_label,
                             model_label,
                             target_feature='aki_72hrs_any',
                             random_state=343,
                             additional_param_dict=dict(),
                             save_model_dir=''):
    X_train = train_df[feature_set].to_numpy()
    X_test = test_df[feature_set].to_numpy()
    
    y_train = train_df[target_feature].to_numpy()
    y_test = test_df[target_feature].to_numpy()

    model = model_callable(**params_dict)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)

    performance_results = {
        'cutpoint_50': cutpoint_analysis.get_results_at_cutpoint(y_test, 
                                                                 y_score[:,1],
                                                                 cutpoint=0.5,
                                                                 max_iteration_count=10000)['metrics'],
        'cutpoint_90': cutpoint_analysis.get_results_at_cutpoint(y_test, 
                                                                 y_score[:,1],
                                                                 cutpoint=0.9,
                                                                 max_iteration_count=10000)['metrics']
    }

    full_metrics = {
        'accuracy_cp50': performance_results['cutpoint_50']['accuracy'],
        'tpr_cp50': performance_results['cutpoint_50']['tpr'],
        'tnr_cp50': performance_results['cutpoint_50']['tnr'],
        'precision_cp50': performance_results['cutpoint_50']['precision'],
        'accuracy_cp90': performance_results['cutpoint_90']['accuracy'],
        'tpr_cp90': performance_results['cutpoint_90']['tpr'],
        'tnr_cp90': performance_results['cutpoint_90']['tnr'],
        'precision_cp90': performance_results['cutpoint_90']['precision'],
        'auroc': roc_auc_score(y_test, y_score[:,1]),
        'auprc': average_precision_score(y_test, y_score[:,1])
    }
    
    if len(save_model_dir) > 0:
        if not os.path.isdir(save_model_dir):
            temp_path = ''
            for subfolder in save_model_dir.split('/')[:-1]:
                temp_path += subfolder + '/'
                if not os.path.isdir(temp_path):
                    os.mkdir(temp_path)
        with open(save_model_dir, 'wb') as outfile:
            pickle.dump(model, outfile)

    return full_metrics

# Define dictionaries of model callables and parameters

In [11]:
model_callable_dict = {
    'Logistic Regression': LogisticRegression,    
    'SVC (rbf)': SVC,
    'SVC (linear)': SVC,
    'SVC (poly)': SVC,
    'Random Forest': RandomForestClassifier,
    'DecisionTree': DecisionTreeClassifier,
    'Gradient Boosting Classifier': GradientBoostingClassifier,
    'Naive Bayes (Gaussian)': GaussianNB
}

model_params_dict = {
    'Logistic Regression': {
        'class_weight': 'balanced',
        'max_iter': 100000,
        'random_state': 343
    },    
    'SVC (rbf)': {
        'probability': True,
        'max_iter': -1,
        'class_weight': 'balanced',
        'random_state': 343,
        'kernel': 'rbf'
    },
    'SVC (linear)': {
        'probability': True,
        'max_iter': -1,
        'class_weight': 'balanced',
        'random_state': 343,
        'kernel': 'linear'
    },
    'SVC (poly)': {
        'probability': True,
        'max_iter': -1,
        'class_weight': 'balanced',
        'random_state': 343,
        'kernel': 'poly'
    },
    'Random Forest': {
        'random_state': 343,
        'class_weight': 'balanced'
    },
    'DecisionTree': {
        'random_state': 343,
        'class_weight': 'balanced'
    },
    'Gradient Boosting Classifier': {
        'random_state': 343
    },
    'Naive Bayes (Gaussian)': dict()
}

# Iterate over models and log performance

In [12]:
for feature_set_key in ll_feature_set_dict.keys():
    if feature_set_key.replace('ll', 'med') not in med_feature_set_dict.keys():
        print(feature_set_key)

In [13]:
for feature_set_key in med_feature_set_dict.keys():
    if feature_set_key.replace('med', 'll') not in ll_feature_set_dict.keys():
        print(feature_set_key)

In [14]:
for feature_set_key in ll_feature_set_dict.keys():
    print(feature_set_key)

ll_roc_logreg_only_intersect
ll_roc_logreg_only_union
ll_prc_logreg_only_intersect
ll_prc_logreg_only_union
ll_roc_svc_intersect
ll_roc_svc_union
ll_prc_svc_intersect
ll_prc_svc_union


# Train models

In [2]:
results_dd_ll = dict()
results_dd_med = dict()

allow_overwrite = True

for ll_feature_set_key in ll_feature_set_dict.keys():
    med_feature_set_key = ll_feature_set_key.replace('ll_', 'med_')
    print('~'*25)
    print('='*20)
    print(ll_feature_set_key.replace('ll_', ''))
    print('='*20)
    feature_set_ll = ll_feature_set_dict[ll_feature_set_key]
    feature_set_med = med_feature_set_dict[med_feature_set_key]
    metric_label = feature_set_key.split('_')[1]
    results_dd_ll[ll_feature_set_key] = dict()
    results_dd_med[med_feature_set_key] = dict()
    
    for model_key in model_callable_dict.keys():
        print(model_key)
        # train_df = train_df_dict[feature_set_key]
        # test_df = test_df_dict[feature_set_key]
        model_callable = model_callable_dict[model_key]
        params_dict = model_params_dict[model_key]
        
        ll_results_filename = 'pickle/final_model_performance_results/' + \
            ll_feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '.pickle'
        
        med_results_filename = 'pickle/final_model_performance_results/' + \
            med_feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '.pickle'
        
        ll_model_filename = 'pickle/models/' + ll_feature_set_key + '_' + model_key.replace(' ', '_') + '.pickle'
        
        med_model_filename = 'pickle/models/' + med_feature_set_key + '_' + model_key.replace(' ', '_') + '.pickle'
        
        if not os.path.isfile(ll_results_filename) or allow_overwrite:
            print('[latest lab imputation]')
            results_dd_ll[ll_feature_set_key][model_key] = assess_model_on_features(
                train_cohort_ll,
                val_cohort_ll,
                feature_set_ll,
                model_callable,
                params_dict,
                ll_feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343,
                save_model_dir=ll_model_filename
            )

            with open(ll_results_filename, 'wb') as outfile:
                pickle.dump(results_dd_ll[ll_feature_set_key][model_key], outfile)
        
        try:
            if not os.path.isfile(med_results_filename) or allow_overwrite:
                print('[median imputation]')
                results_dd_med[med_feature_set_key][model_key] = assess_model_on_features(
                    train_cohort_med,
                    val_cohort_med,
                    feature_set_med,
                    model_callable,
                    params_dict,
                    med_feature_set_key,
                    metric_label,
                    model_key,
                    target_feature='aki_72hrs_any',
                    random_state=343,
                    save_model_dir=med_model_filename
                )

                with open(med_results_filename, 'wb') as outfile:
                    pickle.dump(results_dd_med[med_feature_set_key][model_key], outfile)
        except:
            print('ERROR on median imputation version with intended filename ' + med_results_filename)

In [ ]:
ll_results_dl = []

for fs_key in results_dd_ll.keys():
    for model_key in results_dd_ll[fs_key].keys():
        temp_dict = {
            'feature_set': fs_key,
            'model': model_key
        }
        
        for metric_key in results_dd_ll[fs_key][model_key].keys():
            temp_dict[metric_key] = results_dd_ll[fs_key][model_key][metric_key]
            
        ll_results_dl.append(temp_dict)
        
ll_results_df = pd.DataFrame(ll_results_dl)

ll_results_df = ll_results_df.rename(
    columns={
        'precision_cp50': 'ppv_cp50',
        'precision_cp90': 'ppv_cp90'
    }
)

ll_results_df[[
    'feature_set', 'model', 'ppv_cp90', 'auroc', 'auprc'
]].sort_values('ppv_cp90', ascending=False)

In [ ]:
results_dd_ll = dict()
results_dd_med = dict()

for ll_feature_set_key in ll_feature_set_dict.keys():
    med_feature_set_key = ll_feature_set_key.replace('ll_', 'med_')
    print('~'*25)
    print('='*20)
    print(feature_set_key.replace('ll_', ''))
    print('='*20)
    feature_set_ll = ll_feature_set_dict[ll_feature_set_key]
    feature_set_med = med_feature_set_dict[med_feature_set_key]
    metric_label = feature_set_key.split('_')[1]
    results_dd_ll[ll_feature_set_key] = dict()
    results_dd_med[med_feature_set_key] = dict()
    
    for model_key in model_callable_dict.keys():
        print(model_key)
        # train_df = train_df_dict[feature_set_key]
        # test_df = test_df_dict[feature_set_key]
        model_callable = model_callable_dict[model_key]
        params_dict = model_params_dict[model_key]
        
        ll_results_filename = 'pickle/final_model_performance_results/' + \
            ll_feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '.pickle'
        
        med_results_filename = 'pickle/final_model_performance_results/' + \
            med_feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '.pickle'
        
        if not os.path.isfile(ll_results_filename):
            print('[latest lab imputation]')
            results_dd_ll[ll_feature_set_key][model_key] = assess_model_on_features(
                train_cohort_ll,
                val_cohort_ll,
                feature_set_ll,
                model_callable,
                params_dict,
                ll_feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343)
            
            with open(ll_results_filename, 'wb') as outfile:
                pickle.dump(results_dd_ll[ll_feature_set_key][model_key], outfile)
        
        if not os.path.isfile(med_results_filename):
            print('[median imputation]')
            results_dd_med[med_feature_set_key][model_key] = assess_model_on_features(
                train_cohort_med,
                val_cohort_med,
                feature_set_med,
                model_callable,
                params_dict,
                med_feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343)
            
            with open(med_results_filename, 'wb') as outfile:
                pickle.dump(results_dd_med[med_feature_set_key][model_key], outfile)

In [ ]:
with open('pickle/final_model_performace_results/model_performance_ll_all_types_dd.pickle', 'wb') as outfile:
    pickle.dump(results_dd_ll, outfile)
    
with open('pickle/final_model_performace_results/model_performance_med_all_types_dd.pickle', 'wb') as outfile:
    pickle.dump(results_dd_med, outfile)